In [96]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg, udf
from pyspark.sql.types import StringType, IntegerType
import time

In [97]:
spark = SparkSession.builder.appName("Partition_Practice").getOrCreate()

In [61]:
df = spark.range(0,1000000).withColumn("value", col("id") % 1000)

In [62]:
df.count()

1000000

In [63]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [64]:
print("Before partitions:", df.rdd.getNumPartitions())

Before partitions: 8


In [65]:
df.write.mode("overwrite").csv("output/BeforePart", header = True)

In [66]:
df_repartitioned = df.repartition(20)

In [67]:
print("After partitions:", df_repartitioned.rdd.getNumPartitions())

[Stage 37:>                                                         (0 + 8) / 8]

After partitions: 20


[Stage 37:=============================>                            (4 + 4) / 8]

In [68]:
df_repartitioned.write.mode("overwrite").csv("output/afterPart", header = True)

In [69]:
df_coal = df_repartitioned.coalesce(2)
print("After coalesced:", df_coal.rdd.getNumPartitions())

[Stage 41:==============>                                           (2 + 6) / 8]

After coalesced: 2


In [ ]:
# Optimizatiom & caching

In [70]:
optimized_df = df.filter(col("value") > 500).filter(col("id") < 5000000).select("id","value")

In [71]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#70L, (id#70L % 1000) AS value#72L]
+- *(1) Filter (((id#70L % 1000) > 500) AND (id#70L < 5000000))
   +- *(1) Range (0, 1000000, step=1, splits=8)




In [78]:
start_time = time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. Optimized Execution  | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution  | Count: 499000 | Time: 1.2446 seconds


In [79]:
optimized_df.cache() 

26/06/13 05:57:44 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [82]:
start_time = time.time()
count_second_cache = optimized_df.count()   
end_time = time.time()
print(f"2. Second Optimized Execution  | Count: {count_second_cache} | Time: {round(end_time - start_time, 4)} seconds")

2. Second Optimized Execution  | Count: 499000 | Time: 0.7695 seconds


In [83]:
start_time = time.time()
count_third_cache = optimized_df.count()   
end_time = time.time()
print(f"3. Third Optimized Execution  | Count: {count_third_cache} | Time: {round(end_time - start_time, 4)} seconds")

3. Third Optimized Execution  | Count: 499000 | Time: 0.3129 seconds


In [ ]:
# unpersist

In [85]:
optimized_df.unpersist()
print("\n--> Unpersisted DataFrame to free up memory.")


--> Unpersisted DataFrame to free up memory.


In [ ]:
# UDF = user define function

In [114]:
data = [("Anu", 13), ("Divya", 20), ("Disha", 10)]
df1 = spark.createDataFrame(data, ["Name","Age"])

In [115]:
def categorized_age(age):
    if(age >= 18):
        return "Can Drive"
    elif (age >= 13):
        return "Can drive with licence"
    else:
        return "Cannot drive"

In [116]:
age_category_udf = udf(categorized_age, StringType())

In [117]:
df_method1 = df1.withColumn("Category", age_category_udf(col("Age")))
print("Method 1: Standard UDF via DataFrame API") 
df_method1.show()

Method 1: Standard UDF via DataFrame API


+-----+---+--------------------+
| Name|Age|            Category|
+-----+---+--------------------+
|  Anu| 13|Can drive with li...|
|Divya| 20|           Can Drive|
|Disha| 10|        Cannot drive|
+-----+---+--------------------+



In [119]:
spark.udf.register("sql_categorized_age",categorized_age, StringType())

df1.createOrReplaceTempView("people") 

In [124]:
sql_df = spark.sql('''
    select 
        Name, Age,
        sql_categorized_age(Age) AS Category
    from people
''')

In [125]:
sql_df.show()

+-----+---+--------------------+
| Name|Age|            Category|
+-----+---+--------------------+
|  Anu| 13|Can drive with li...|
|Divya| 20|           Can Drive|
|Disha| 10|        Cannot drive|
+-----+---+--------------------+



In [126]:
spark.stop()